# NB2 · Classificazione, regolarizzazione e cross-validation

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB2_classificazione_regolarizzazione.ipynb)

**logistica su Default, la soglia, ridge e lasso, la scelta di lambda**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**. esegui le celle in ordine, dall'alto verso il basso.

## 1. Dati: chi sara' insolvente?

dataset **Default** di ISL: per 10.000 clienti, se sono andati in insolvenza (`default`), se sono studenti (`student`), il saldo della carta (`balance`) e il reddito (`income`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv("https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/Default.csv")
df["default"] = (df["default"] == "Yes").astype(int)
df["student"] = (df["student"] == "Yes").astype(int)
print("tasso di insolvenza:", round(df["default"].mean(), 3))
df.head()

## 2. Regressione logistica

prevediamo la probabilita' di insolvenza dal saldo, dal reddito e dallo stato di studente.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

X = df[["balance", "income", "student"]].values
y = df["default"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)

logit = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
prob = logit.predict_proba(Xte)[:, 1]
print(classification_report(yte, (prob > 0.5).astype(int), target_names=["non insolvente", "insolvente"]))

## 3. La soglia e' una scelta, non un dato

di default si classifica "insolvente" se la probabilita' supera 0,5. ma la soglia si puo' spostare: abbassarla trova piu' insolventi (piu' veri positivi) ma con piu' falsi allarmi.

> **manopola**: cambia `SOGLIA` (prova 0.2, 0.3, 0.5) e guarda come cambia la matrice di confusione.

In [ ]:
# >>> MANOPOLA: soglia di decisione <<<
SOGLIA = 0.5

pred = (prob > SOGLIA).astype(int)
cm = confusion_matrix(yte, pred)
print("matrice di confusione (righe = vero, colonne = previsto):")
print(pd.DataFrame(cm, index=["vero: no", "vero: sì"], columns=["prev: no", "prev: sì"]))
veri_pos = cm[1, 1]; falsi_neg = cm[1, 0]
print(f"\ninsolventi individuati: {veri_pos} su {veri_pos + falsi_neg}")

## 4. Regolarizzazione: ridge e lasso

con molti predittori conviene **penalizzare** i coefficienti. in `LogisticRegression` la forza della penalita' e' `C = 1/lambda`: **C piccolo = penalita' forte** (coefficienti piu' piccoli).

> **manopola**: cambia `C` (prova 0.001, 0.01, 1, 100) e osserva come cambiano i coefficienti.

In [ ]:
# >>> MANOPOLA: forza della regolarizzazione (C = 1/lambda) <<<
C = 1.0

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

ridge = LogisticRegression(penalty="l2", C=C, max_iter=2000).fit(Xtr_s, ytr)
nomi = ["balance", "income", "student"]
print(f"C = {C}  (penalità L2 / ridge)")
for n, b in zip(nomi, ridge.coef_[0]):
    print(f"  coef {n:>8}: {b:+.3f}")

il **lasso** (penalita' L1) puo' azzerare del tutto i coefficienti meno utili, facendo selezione delle variabili.

In [ ]:
lasso = LogisticRegression(penalty="l1", C=0.05, solver="liblinear", max_iter=2000).fit(Xtr_s, ytr)
print("coefficienti con il lasso (C piccolo):")
for n, b in zip(nomi, lasso.coef_[0]):
    stato = "  <- azzerato" if abs(b) < 1e-6 else ""
    print(f"  {n:>8}: {b:+.3f}{stato}")

## 5. Scegliere lambda con la cross-validation

non scegliamo `C` a occhio: lo scegliamo con la k-fold cross-validation, che stima l'errore su dati nuovi.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

cv = LogisticRegressionCV(Cs=np.logspace(-3, 2, 20), cv=5, max_iter=2000).fit(Xtr_s, ytr)
print("C scelto dalla cross-validation:", round(cv.C_[0], 4))
print("accuratezza sul test:", round(cv.score(Xte_s, yte), 3))

---

### Cella bonus: il "percorso" del lasso

al crescere della penalita', i coefficienti del lasso si restringono e a uno a uno vanno a zero.

In [ ]:
# BONUS
Cs = np.logspace(-3, 1, 30)
percorso = []
for c in Cs:
    m = LogisticRegression(penalty="l1", C=c, solver="liblinear", max_iter=2000).fit(Xtr_s, ytr)
    percorso.append(m.coef_[0])
percorso = np.array(percorso)

plt.figure(figsize=(7.5, 5))
for j, n in enumerate(nomi):
    plt.plot(Cs, percorso[:, j], "o-", label=n)
plt.xscale("log"); plt.xlabel("C = 1/lambda (log)"); plt.ylabel("coefficiente")
plt.title("Percorso del lasso: i coefficienti si restringono"); plt.legend(); plt.axhline(0, color="grey", lw=1)
plt.show()